# Sentinel — Predictive Maintenance: Data & Model Overview

**One notebook, end to end:** data description → pipeline → feature engineering → the two models →
accuracy (precision & recall) → business view.

This project answers two questions for a fleet of 100 machines:

| Question | Model | Output |
|---|---|---|
| Will a machine **fail within 12 hours**? | XGBoost classifier | failure probability 0–1 |
| **When** should each machine be serviced? | Weibull survival + classifier-surrogate recurrence | risk score H(t); service when H ≥ 1 |

Dataset: the Microsoft Azure Predictive Maintenance dataset (100 machines × 1 year, hourly).


## 0 · Setup

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

# Resolve repo root whether run from repo/ or repo/notebooks/
ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists() and (ROOT.parent / "data" / "raw").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
REPORTS = ROOT / "outputs" / "reports"
print("Repo root:", ROOT)
print("Raw data :", RAW, "|", "exists" if RAW.exists() else "MISSING")


---
## 1 · Data Description

Five raw tables. Four are inputs (telemetry, errors, maintenance, machines) and one is the
**answer key** (failures) used only to build labels — never as a model feature.


In [ ]:
tele = pd.read_csv(RAW / "PdM_telemetry.csv", parse_dates=["datetime"])
errs = pd.read_csv(RAW / "PdM_errors.csv", parse_dates=["datetime"])
maint = pd.read_csv(RAW / "PdM_maint.csv", parse_dates=["datetime"])
fails = pd.read_csv(RAW / "PdM_failures.csv", parse_dates=["datetime"])
machines = pd.read_csv(RAW / "PdM_machines.csv")

def describe(name, df, unit):
    dt = df["datetime"] if "datetime" in df.columns else None
    return {
        "table": name, "rows": len(df), "cols": df.shape[1],
        "one row =": unit,
        "date range": f"{dt.min():%Y-%m-%d} to {dt.max():%Y-%m-%d}" if dt is not None else "static",
    }

summary = pd.DataFrame([
    describe("telemetry", tele, "one machine, one hour"),
    describe("errors", errs, "a non-fatal warning event"),
    describe("maintenance", maint, "a component replacement"),
    describe("failures", fails, "an actual breakdown (LABEL)"),
    describe("machines", machines, "static machine info"),
])
summary


### Column dictionary

- **telemetry** — `datetime, machineID, volt, rotate, pressure, vibration` (4 hourly sensors)
- **errors** — `datetime, machineID, errorID` (error1–5): non-fatal warnings
- **maintenance** — `datetime, machineID, comp` (comp1–4): a component was replaced
- **failures** — `datetime, machineID, failure` (comp1–4): a real breakdown → **the label source**
- **machines** — `machineID, model` (model1–4 = "Type 1–4"), `age` (years)

> **Part vs Type:** *Type* = the machine model (model1–4). *Part* = the component (comp1–4). Each
> machine has 4 independently-replaceable components.


In [ ]:
print("Telemetry sample:")
display(tele.head(3))
print("\nSensor summary statistics:")
display(tele[["volt","rotate","pressure","vibration"]].describe().round(2))


In [ ]:
# Failures per component — the rare positive class
fc = fails["failure"].value_counts().sort_index()
print("Total failures:", len(fails), "across", fails["machineID"].nunique(), "machines")
print("Failure rate  :", f"{len(fails)/len(tele)*100:.3f}% of all machine-hours (highly imbalanced)")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
fc.plot.bar(ax=ax[0], color="#f43f5e"); ax[0].set_title("Failures per component"); ax[0].set_ylabel("count")
errs["errorID"].value_counts().sort_index().plot.bar(ax=ax[1], color="#fb923c"); ax[1].set_title("Errors per type")
maint["comp"].value_counts().sort_index().plot.bar(ax=ax[2], color="#6366f1"); ax[2].set_title("Replacements per component")
for a in ax: a.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()


In [ ]:
# Sensor distributions + failures over time
fig, ax = plt.subplots(2, 2, figsize=(14, 6))
for a, s, c in zip(ax.ravel(), ["volt","rotate","pressure","vibration"], ["#6366f1","#0ea5e9","#10b981","#f43f5e"]):
    tele[s].hist(bins=60, ax=a, color=c, alpha=0.8); a.set_title(f"{s} distribution"); a.grid(alpha=0.2)
plt.tight_layout(); plt.show()

fpm = fails.set_index("datetime").resample("W").size()
plt.figure(figsize=(14, 3))
fpm.plot(color="#f43f5e"); plt.title("Failures over time (weekly)"); plt.ylabel("failures"); plt.grid(alpha=0.2)
plt.tight_layout(); plt.show()


### Machine fleet
100 machines across 4 model types with varying ages.

In [ ]:
display(machines.groupby("model").agg(machines=("machineID","count"), avg_age=("age","mean")).round(1))
machines["age"].hist(bins=20, figsize=(8,2.6), color="#6366f1"); plt.title("Machine age (years)"); plt.grid(alpha=0.2); plt.show()


---
## 2 · Process — block-level representation

```
                      data/raw/ (5 CSVs)
                           |
              +------------+------------+
              |  feature engineering    |   src/pdm/features.py
              |  ~76 features/machine-hr|   rolling stats, error counts, part age
              +------------+------------+
                           |
        +------------------+-------------------+
        |                                      |
 +------v------+                        +------v---------------------+
 | LABELS      |  src/pdm/labels.py     | RENEWAL LIVES              | survival.py
 | fail<=12h?  |                        | per (machine, component)   |
 +------+------+                        +------+---------------------+
        |                                      |
 +------v----------------+          +----------v-------------------+
 | CLASSIFICATION        |          | RISK SCORE (PdM)             |
 | XGBoost, 20 features  |          | Weibull on censored lives    | maintenance.py
 | Optuna tuning         |          |   -> characteristic life     |
 | -> P(fail in 12h)     |          | Weibull on classifier alarms | surrogate.py
 +------+----------------+          |   -> recurrence risk         |
        |                           +----------+-------------------+
        |                                      |
        +------------------+-------------------+
                           |
                   api/ (FastAPI)  +  frontend/ (Next.js)
                 Classification page    Risk Score page
```

**Offline (run once):** `train.py` → `scripts/score_dataset.py` → `scripts/build_rul.py` produce the
artifacts in `outputs/`. **Online:** the API re-scores any chosen "as-of" hour and the dashboard renders it.


---
## 3 · Feature Engineering

Raw readings alone are weak; **trends and context** carry the signal. ~76 features are engineered,
then pruned to the **top 20** (gain + SHAP importance).


In [ ]:
feat = json.loads((REPORTS / "selected_features.json").read_text())
fam = {"hours_since_": "component age (recency of replacement)",
       "error": "rolling error counts",
       "volt": "voltage rolling stats", "rotate": "rotation rolling stats",
       "pressure": "pressure rolling stats", "vibration": "vibration rolling stats",
       "model_": "machine type (one-hot)"}
def family(f):
    for k, v in fam.items():
        if f.startswith(k) or k in f: return v
    return "other"
sel = pd.DataFrame({"feature": feat}); sel["family"] = sel["feature"].map(family)
print(f"{len(feat)} selected features")
display(sel)
print("\nBy family:"); display(sel["family"].value_counts())


**Feature families**
- **Rolling sensor stats** — mean/std/min/max of each sensor over 3h / 12h / 24h windows (trend, instability)
- **Rolling error counts** — error1–5 counts over the same windows (error4 is the most predictive)
- **Component age** — `hours_since_comp1..4` via `merge_asof` (the dominant predictor family)
- **Machine metadata** — model one-hot + age

**Labels** (`src/pdm/labels.py`): `label = 1` if any failure occurs in the next **12 hours**, else 0.
Features look *backward*; labels look *forward* — the separation that prevents leakage.


---
## 4 · Classification model — XGBoost

Gradient-boosted trees on the 20 features, tuned with **Optuna** (50 trials), class imbalance
handled with `scale_pos_weight`. Chronological 80/20 split (never shuffled).


In [ ]:
summ = json.loads((REPORTS / "summary.json").read_text())
best = json.loads((REPORTS / "best_params.json").read_text()) if (REPORTS/"best_params.json").exists() else summ.get("best_params", {})
print("Horizon (hours)      :", summ["horizon_hours"])
print("Final features       :", summ["n_features_final"])
print("Best hyperparameters :"); [print(f"   {k}: {v}") for k, v in best.items()]


---
## 5 · Accuracy Evaluation — Precision & Recall

**Why AUC-PR, not accuracy?** With ~0.09% positives, a "predict healthy always" model scores 99.9%
accuracy yet is useless. Precision/Recall/AUC-PR are honest for rare events.

- **Recall** = of real failures, how many we caught = *TP / (TP + FN)*
- **Precision** = of machines we flagged, how many really failed = *TP / (TP + FP)*


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from src.pdm.scoring import chronological_test_split

scored = pd.read_parquet(ROOT / "outputs" / "scored.parquet")
y_true, y_prob = chronological_test_split(scored, 0.20)   # same split used in training
THRESH = 0.5
y_pred = (y_prob >= THRESH).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"Test rows: {len(y_true):,}  |  actual failures (positives): {int(y_true.sum()):,}")
print(f"\nHeadline metrics (threshold = {THRESH}):")
print(f"   AUC-PR  : {summ['auc_pr']:.3f}   (main metric for imbalanced data)")
print(f"   AUC-ROC : {summ['auc_roc']:.3f}")
print(f"   Recall  : {recall_score(y_true, y_pred):.3f}   (failures caught)")
print(f"   Precision: {precision_score(y_true, y_pred):.3f}   (alerts that are right)")
print(f"   F1      : {f1_score(y_true, y_pred):.3f}")


In [ ]:
# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(4.6, 3.8))
im = ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, f"{v:,}", ha="center", va="center",
            color="white" if v > cm.max()/2 else "black", fontsize=12, fontweight="bold")
ax.set_xticks([0,1]); ax.set_xticklabels(["Pred healthy","Pred FAIL"])
ax.set_yticks([0,1]); ax.set_yticklabels(["Actual healthy","Actual FAIL"])
ax.set_title(f"Confusion matrix @ {THRESH}"); plt.tight_layout(); plt.show()


In [ ]:
# Threshold sweep — precision / recall / F1 trade-off
tbl = pd.read_csv(REPORTS / "threshold_table.csv")
plt.figure(figsize=(10, 4))
plt.plot(tbl["threshold"], tbl["recall"], label="Recall (failures caught)", color="#10b981", lw=2)
plt.plot(tbl["threshold"], tbl["precision"], label="Precision (alerts right)", color="#0ea5e9", lw=2)
plt.plot(tbl["threshold"], tbl["f1"], label="F1 (balance)", color="#f59e0b", lw=2, ls="--")
plt.axvline(0.5, color="#94a3b8", ls=":", label="default 0.50")
plt.xlabel("alert threshold"); plt.ylabel("score"); plt.title("Precision / Recall vs threshold")
plt.legend(); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()


In [ ]:
# Per-component performance
comp = pd.read_csv(REPORTS / "component_metrics.csv")
display(comp.round(3))
comp.set_index("component")[["recall","precision"]].plot.bar(
    figsize=(8, 3), color=["#10b981","#0ea5e9"]); plt.title("Per-component recall vs precision")
plt.xticks(rotation=0); plt.ylim(0,1.05); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()


**Reading the trade-off:** the model catches ~**100% of failures** (recall) with precision
that rises as you raise the threshold. Lowering the alert threshold → catch everything but more
false alarms; raising it → fewer, higher-confidence alerts. The **ALERTS** slider in the app is
exactly this threshold.


---
## 6 · Risk Score model — Weibull maintenance cycle + surrogate recurrence

Two survival models power the "when to service" view:

1. **Part-wear cycle** — a Weibull fitted per component on **renewal lives with censoring** (parts
   that ran long without failing are censored, not discarded — this avoids a short-life bias).
   `H(t) = (age / characteristic-life)^shape`; service when **H ≥ 1**.
2. **Surrogate recurrence** — a Weibull fitted on the **classifier's alarm events** (each predicted
   failure = a surrogate event). Estimates when the next pre-failure state recurs; validated against
   real failures.


In [ ]:
import joblib
try:
    bundle = joblib.load(ROOT / "outputs" / "models" / "survival.joblib")
    from src.pdm.maintenance import comp_weibull_from_dict
    cw = comp_weibull_from_dict(bundle.get("comp_weibull", {}))
    print("Part-wear characteristic life per component (censored Weibull):")
    for c, w in cw.items():
        print(f"   {c}: cycle(lambda)={w.scale:6.1f} days   shape(rho)={w.shape:.2f}   (n={w.n_lives} lives)")
    sv = bundle.get("surrogate_weibull"); val = bundle.get("surrogate_validation", {})
    if sv:
        print(f"\nSurrogate recurrence cycle: {sv['scale']:.1f} days   shape={sv['shape']:.2f}   events={sv['n_events']}")
        print(f"Validation vs real failures: recall={val.get('recall')}  precision={val.get('precision')}  median_lead={val.get('median_lead_hours')}h")
except Exception as e:
    print("Run scripts/build_rul.py first to generate survival.joblib —", e)


**Decision rule:** a machine is due for service when its risk score **H(t) reaches 1.0** — the
part has reached its characteristic life — OR when the classifier is flagging an imminent failure
right now. H resets to 0 each time a part is replaced (the sawtooth).


---
## 7 · Business View

Predictive maintenance turns **unplanned breakdowns** (expensive, unsafe) into **planned service**
(cheap, scheduled). The value = failures prevented − cost of acting early.


In [ ]:
# Illustrative cost model (edit the assumptions to your plant's economics)
COST_UNPLANNED = 10_000   # $ per unplanned failure (downtime, damage, overtime)
COST_PLANNED   = 1_500    # $ per planned service
COST_FALSE_ALARM = 300    # $ per unnecessary inspection

n_fail = int(y_true.sum())
recall = recall_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
caught = int(round(recall * n_fail))
missed = n_fail - caught
false_alarms = int(round(caught * (1 - precision) / max(precision, 1e-9)))

reactive = n_fail * COST_UNPLANNED
predictive = caught * COST_PLANNED + missed * COST_UNPLANNED + false_alarms * COST_FALSE_ALARM
savings = reactive - predictive

print("=== Illustrative economics over the test period ===")
print(f"Real failures in period        : {n_fail}")
print(f"Caught early by the model       : {caught}  ({recall*100:.0f}% recall)")
print(f"Missed (still unplanned)        : {missed}")
print(f"False alarms (extra inspections): ~{false_alarms}")
print(f"\nReactive (run-to-failure) cost  : $ {reactive:,.0f}")
print(f"Predictive maintenance cost     : $ {predictive:,.0f}")
print(f"Estimated savings               : $ {savings:,.0f}  ({savings/reactive*100:.0f}% reduction)")


In [ ]:
# Maintenance-strategy comparison
strat = pd.DataFrame({
    "strategy": ["Reactive (run to failure)", "Scheduled (fixed calendar)", "Predictive (this system)"],
    "catches failures early": ["No", "Some", "~100% (recall)"],
    "unnecessary service": ["None", "High (over-services)", "Low (precision-controlled)"],
    "relative cost": ["Highest", "Medium", "Lowest"],
})
display(strat)

plt.figure(figsize=(6,3.2))
plt.bar(["Reactive","Predictive"], [reactive, predictive], color=["#f43f5e","#10b981"])
plt.title("Total maintenance cost (illustrative)"); plt.ylabel("$"); plt.grid(alpha=0.2, axis="y")
plt.tight_layout(); plt.show()


### Business takeaways
- **Two complementary signals.** The classifier catches **acute** failures (sensor spikes, ~100%
  recall); the Weibull risk score schedules **wear-out** service before parts reach end of life.
- **Tunable sensitivity.** The alert threshold trades false alarms against missed failures — set it
  to your cost ratio (cheap inspections → lower threshold; expensive inspections → higher).
- **Fleet planning.** The risk score turns the fleet into a ranked worklist ("service now / soon /
  planned"), enabling scheduled downtime instead of emergency repairs.
- **Auditable & explainable.** Every alert has SHAP attribution (which sensor drove it); every
  service date traces to a fitted survival cycle validated against real failures.


---
## How to reproduce
```bash
pip install -r requirements.txt
python train.py --config config.yaml                  # classifier + reports
python scripts/score_dataset.py --config config.yaml  # scored.parquet
python scripts/build_rul.py --config config.yaml       # survival + surrogate + rul
```
Then run this notebook top to bottom.
